# 02 · Baselines
Fase 2 del plan: popularidad, recompra y filtrado colaborativo (ALS), evaluados con **MAP@12** y **recall@12** sobre la última semana.

**Input requerido:** la salida del notebook `01_eda` (agrégala con *Add Input → Your Work*).

In [1]:
!pip install -q implicit

In [2]:
import pandas as pd, numpy as np, glob, os
from pathlib import Path
import scipy.sparse as sp

# Busca los parquet del notebook 01 sin depender del nombre exacto del input
hits = glob.glob("/kaggle/input/**/train_small.parquet", recursive=True)
assert hits, "No encontré train_small.parquet: agrega la salida de 01_eda como Input"
IN = Path(hits[0]).parent
OUT = Path("/kaggle/working")
print("Leyendo de", IN)

train = pd.read_parquet(IN/"train_small.parquet")   # últimas 12 semanas antes de validación
val   = pd.read_parquet(IN/"val.parquet")
print(train.shape, val.shape, train.t_dat.min().date(), "→", train.t_dat.max().date())

Leyendo de /kaggle/input/notebooks/valentina3030/01-eda-dmii
(3664080, 5) (240311, 5) 2020-06-24 → 2020-09-15


## Métricas
MAP@12 como en la competencia: promedio sobre los clientes que compraron algo en la semana de validación.

In [3]:
def apk(actual, predicted, k=12):
    if not actual:
        return 0.0
    predicted = predicted[:k]
    hits, score, seen = 0, 0.0, set()
    for i, p in enumerate(predicted):
        if p in actual and p not in seen:
            hits += 1
            score += hits / (i + 1)
        seen.add(p)
    return score / min(len(actual), k)

def evaluate(preds, gt, k=12):
    """preds: dict cliente -> lista de article_id; gt: dict cliente -> set de article_id."""
    maps, recs = [], []
    for c, actual in gt.items():
        p = preds.get(c, [])
        maps.append(apk(actual, p, k))
        recs.append(len(actual & set(p[:k])) / len(actual))
    return {"MAP@12": np.mean(maps), "recall@12": np.mean(recs)}

gt = val.groupby("customer_id").article_id.agg(set).to_dict()
val_customers = list(gt.keys())
print("Clientes en validación:", len(gt))
results = {}

Clientes en validación: 68984


## Baseline 1 · Popularidad reciente
Los 12 artículos más vendidos en la última semana de train, iguales para todos.

In [4]:
last_week = train[train.t_dat > train.t_dat.max() - pd.Timedelta(days=7)]
top12 = last_week.article_id.value_counts().head(12).index.tolist()

preds_pop = {c: top12 for c in val_customers}
results["Popularidad (última semana)"] = evaluate(preds_pop, gt)
results["Popularidad (última semana)"]

{'MAP@12': np.float64(0.008747681312782597),
 'recall@12': np.float64(0.02550428716366145)}

## Baseline 2 · Recompra + popularidad
En H&M la gente recompra mucho, así que este baseline suele ser difícil de superar. Recomienda lo que el cliente compró más recientemente y completa con popularidad.

In [5]:
recent = (train.sort_values("t_dat", ascending=False)
               .drop_duplicates(["customer_id", "article_id"])
               .groupby("customer_id").article_id.agg(list))

def fill(lst, k=12):
    out = list(dict.fromkeys(lst))[:k]
    out += [a for a in top12 if a not in out][:k - len(out)]
    return out

preds_rep = {c: fill(recent.get(c, [])) for c in val_customers}
results["Recompra + popularidad"] = evaluate(preds_rep, gt)
results["Recompra + popularidad"]

{'MAP@12': np.float64(0.025970409849419035),
 'recall@12': np.float64(0.052510706444840835)}

## Baseline 3 · Filtrado colaborativo (ALS con `implicit`)
Matriz cliente × artículo con el número de compras como señal implícita. Clientes sin historial en train reciben popularidad.

In [6]:
# Índices enteros para clientes y artículos
u_codes, u_uniques = pd.factorize(train.customer_id)
i_codes, i_uniques = pd.factorize(train.article_id)
user_items = sp.csr_matrix(
    (np.ones(len(train), dtype=np.float32), (u_codes, i_codes)),
    shape=(len(u_uniques), len(i_uniques)),
)
user_items.sum_duplicates()
print(user_items.shape, "nnz:", user_items.nnz)

from implicit.nearest_neighbours import bm25_weight
weighted = bm25_weight(user_items, K1=100, B=0.8).tocsr()

(505310, 41376) nnz: 3237697


In [7]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(factors=128, regularization=0.05, iterations=15, random_state=42)
als.fit(weighted)

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

In [8]:
uid_of = pd.Series(np.arange(len(u_uniques)), index=u_uniques)
known = [c for c in val_customers if c in uid_of.index]
uids = uid_of.loc[known].values
print(f"Clientes de validación con historial: {len(known)} / {len(val_customers)}")

preds_als = {c: top12 for c in val_customers}   # fallback para clientes nuevos
B = 50_000
for s in range(0, len(uids), B):
    batch = uids[s:s+B]
    ids, _ = als.recommend(batch, user_items[batch], N=12, filter_already_liked_items=False)
    for c, row in zip(known[s:s+B], ids):
        preds_als[c] = i_uniques[row].tolist()

results["ALS (implicit)"] = evaluate(preds_als, gt)
results["ALS (implicit)"]

Clientes de validación con historial: 49219 / 68984


{'MAP@12': np.float64(0.015121737068685415),
 'recall@12': np.float64(0.03729694298560386)}

## Tabla de resultados
Este es el punto de referencia para todas las fases siguientes.

In [9]:
res = pd.DataFrame(results).T.sort_values("MAP@12", ascending=False)
res.to_csv(OUT/"baselines.csv")
res.style.format("{:.4f}")

,MAP@12,recall@12
Recompra + popularidad,0.0260,0.0525
ALS (implicit),0.0151,0.0373
Popularidad (última semana),0.0087,0.0255


## Guardar artefactos para fases siguientes
Los factores de ALS servirán como señal de personalización en la Fase 5.

In [10]:
np.save(OUT/"als_item_factors.npy", als.item_factors)
np.save(OUT/"als_user_factors.npy", als.user_factors)
pd.Series(i_uniques).to_frame("article_id").to_parquet(OUT/"als_items.parquet")
pd.Series(u_uniques).to_frame("customer_id").to_parquet(OUT/"als_users.parquet")
print(os.listdir(OUT))

['als_item_factors.npy', '__notebook__.ipynb', 'als_items.parquet', 'als_user_factors.npy', 'als_users.parquet', 'baselines.csv']
